一个简单的对话机器人，包含消息裁剪和流式输出

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from dotenv import load_dotenv

load_dotenv(override=True)

MODEL_NAME = "deepseek-v4-flash"
MAX_PAIRS_HISTORY = 10
EXIT_KEYWORD = "exit"


def keep_recent_messages(messages, max_pairs_history=MAX_PAIRS_HISTORY):
    """
    保持最近的对话历史，最多保留 max_pairs_history 条消息。
    :param messages: 包含用户和助手消息的列表。
    :param max_pairs_history: 最大保留的消息对数。
    :return: 保留后的消息列表。
    """
    system_messages = [m for m in messages if isinstance(m, SystemMessage)]
    conversation_messages = [m for m in messages if not isinstance(m, SystemMessage)]

    return system_messages + conversation_messages[-(2 * max_pairs_history):]

model = init_chat_model(MODEL_NAME) # 常见模型在.env有配置信息可以直接用模型名称创建

messages = [
    SystemMessage(content="你是一个Python专业智能助手，叫做小P，会用通俗易懂的语言回答用户的问题"),
]

print(f"⭐️请输入问题,输入{EXIT_KEYWORD}退出:")
i = 1
while True:

    print(f"-----------------第{i}轮对话开始\n")
    user_input = input()
    if user_input.lower() == EXIT_KEYWORD.lower():
        print("对话结束")
        break

    messages.append(HumanMessage(content=user_input))
    
    print("小P助手：",end = "",flush=True)

    ai_reply = ""
    recent_messages = keep_recent_messages(messages, max_pairs_history=MAX_PAIRS_HISTORY)

    for chunk in model.stream(recent_messages):
        if chunk.content:
            print(chunk.content, end="", flush=True)
            ai_reply += chunk.content
    
    print(f"\n-----------------第{i}轮对话结束\n")
    i += 1
    messages.append(AIMessage(content=ai_reply))






⭐️请输入问题,输入exit退出:
-----------------第1轮对话开始

小P助手：Python是一种简单易学、功能强大的编程语言，适合从初学者到专业开发者用于开发各种应用。
-----------------第1轮对话结束

-----------------第2轮对话开始

小P助手：你提到的“Python性能差”其实是在特定场景下的比较。具体来说，它主要表现在以下方面：

1. **解释执行**：Python代码是逐行解释执行的，不像C或C++那样先编译成机器码再运行。解释过程本身带来了额外开销，所以纯计算密集型任务（比如大量循环、数学运算）会比编译型语言慢很多。

2. **全局解释器锁（GIL）**：Python的CPython实现有一个GIL，它确保同一时刻只有一个线程在执行字节码。这意味着在多核CPU上，如果你的程序是CPU密集型的多线程任务，GIL会成为瓶颈，无法充分利用多核并行加速。但如果是I/O密集型（如网络请求、文件读写）或者使用多进程（而不是多线程），GIL的影响就小很多。

3. **动态类型和内存管理**：Python是动态类型语言，运行时需要检查变量类型，并且垃圾回收机制（引用计数+分代回收）也会带来性能开销。相比之下，静态类型语言在编译时就确定了类型，内存管理更高效。

不过，这些“性能差”的结论是有前提的。在实际开发中，Python的优势（开发效率高、生态丰富、库支持强大）往往能掩盖性能上的短板。很多性能瓶颈可以通过以下方式缓解：

- **使用C扩展**：NumPy、Pandas等底层用C编写的库，在数据处理和数值计算上速度快得接近C。
- **用多进程替代多线程**：绕过GIL，利用多核。
- **使用PyPy等JIT编译器**：PyPy可以大幅提升纯Python代码的运行速度。
- **将热点代码用C/C++重写**：通过Cython或ctypes调用外部库。

所以更准确的说法是：Python在极端计算密集或高并发实时系统中性能可能不理想，但在绝大多数日常应用、快速原型开发、数据分析和人工智能领域，它的“性能”在开发效率和可维护性面前往往是可接受的。
-----------------第2轮对话结束

-----------------第3轮对话开始

对话结束
